> **Open in VS Code:** From your cloned `s26-06643` repository, run in the terminal:
> ```bash
> code sse/09-github-actions/09-github-actions.ipynb
> ```
> [View source on GitHub](https://github.com/jkitchin/s26-06643/blob/main/sse/09-github-actions/09-github-actions.ipynb)

# Module 09: CI/CD with GitHub Actions

Automate testing and deployment.

## Learning Objectives

1. Understand CI/CD concepts
2. Create GitHub Actions workflows
3. Use AI to generate workflow files
4. Run tests across platforms

## What is CI/CD?

- **CI (Continuous Integration)**: Automatically test code on every push
- **CD (Continuous Deployment)**: Automatically deploy when tests pass

GitHub Actions runs workflows in response to events (push, PR, schedule).

## Basic Workflow

Create `.github/workflows/test.yml`:

```yaml
name: Tests

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    
    steps:
      - uses: actions/checkout@v4
      
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      
      - name: Install dependencies
        run: |
          pip install -e ".[dev]"
      
      - name: Run tests
        run: pytest --cov
```

## Exercise 1: Your First Workflow (10 min)

Get a GitHub Actions workflow running on your repository.

1. **In your GitHub repo, create the workflow file.** From your local clone:
   ```bash
   mkdir -p .github/workflows
   ```

2. **Create `.github/workflows/hello.yml`** with this content:
   ```yaml
   name: Hello World
   on: push

   jobs:
     greet:
       runs-on: ubuntu-latest
       steps:
         - run: echo "Hello from GitHub Actions!"
         - run: python --version
         - run: pip --version
   ```

3. **Commit and push:**
   ```bash
   git add .github/workflows/hello.yml
   git commit -m "ci: add hello world workflow"
   git push
   ```

4. **Go to the Actions tab** on your GitHub repo page. You should see your workflow running (or already completed).

5. **Click on the run** to see the logs. Answer these questions:
   - Did it pass (green check)?
   - What Python version was available by default?
   - How long did it take?

**Expected outcome:** You have a working GitHub Actions workflow and understand how to view its output.

## AI-Generated Workflows

```
> Create a GitHub Actions workflow that:
  - Runs on push and PR to main
  - Tests on Python 3.10, 3.11, 3.12
  - Tests on Ubuntu, macOS, Windows
  - Runs black and ruff checks
  - Runs pytest with coverage
  - Uploads coverage to Codecov
```

AI generates complete workflow - review for correctness!

## Exercise 2: Add a Test Workflow (10 min)

Create a workflow that actually runs your project's tests.

1. **Create `.github/workflows/tests.yml`:**
   ```yaml
   name: Tests
   on: [push, pull_request]

   jobs:
     test:
       runs-on: ubuntu-latest
       steps:
         - uses: actions/checkout@v4

         - name: Set up Python
           uses: actions/setup-python@v5
           with:
             python-version: '3.11'

         - name: Install package
           run: pip install -e ".[dev]"

         - name: Run tests
           run: pytest
   ```

2. **Commit and push:**
   ```bash
   git add .github/workflows/tests.yml
   git commit -m "ci: add test workflow"
   git push
   ```

3. **Go to the Actions tab** and watch the workflow run.

4. **If it fails** (this is common!), click on the failed job and read the logs. The most common issues:
   - Missing dependencies (add them to your `pyproject.toml` or `requirements.txt`)
   - No `[dev]` extra defined (change to `pip install -e .` and `pip install pytest` separately)
   - Tests that depend on local files or environment variables

5. **Fix and push again.** Each push triggers a new run.

**Expected outcome:** Your tests run automatically on every push. You've debugged at least one CI failure (a rite of passage!).

## Matrix Testing

Test across multiple configurations:

```yaml
jobs:
  test:
    runs-on: ${{ matrix.os }}
    strategy:
      matrix:
        os: [ubuntu-latest, macos-latest, windows-latest]
        python-version: ['3.10', '3.11', '3.12']
    
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - run: pip install -e ".[dev]"
      - run: pytest
```

## Exercise 3: Matrix Build (10 min)

Test your package across multiple Python versions simultaneously.

1. **Modify your `.github/workflows/tests.yml`** to use a matrix strategy:
   ```yaml
   name: Tests
   on: [push, pull_request]

   jobs:
     test:
       runs-on: ubuntu-latest
       strategy:
         matrix:
           python-version: ["3.10", "3.11", "3.12"]

       steps:
         - uses: actions/checkout@v4

         - name: Set up Python ${{ matrix.python-version }}
           uses: actions/setup-python@v5
           with:
             python-version: ${{ matrix.python-version }}

         - name: Install package
           run: pip install -e ".[dev]"

         - name: Run tests
           run: pytest
   ```

   **Important:** The quotes around `"3.10"` matter! Without them, YAML interprets `3.10` as the number `3.1`.

2. **Commit and push:**
   ```bash
   git add .github/workflows/tests.yml
   git commit -m "ci: test across Python 3.10-3.12"
   git push
   ```

3. **Go to the Actions tab.** You should see **3 parallel jobs** — one for each Python version.

4. **Check the results:** Do all Python versions pass? If one fails but others pass, that's valuable information about compatibility.

**Expected outcome:** You see 3 jobs running in parallel on the Actions tab, and you understand how matrix builds help catch version-specific issues.

## Status Badges

Add to README:

```markdown
![Tests](https://github.com/user/repo/actions/workflows/test.yml/badge.svg)
```

Shows build status at a glance.

## Exercise 4: Add a Badge (5 min)

Show your build status right on your README.

1. **Go to your Actions tab** on GitHub.

2. **Click on your "Tests" workflow** in the left sidebar.

3. **Click the `...` menu** (top right) and select **"Create status badge"**.

4. **Copy the markdown** it generates. It will look something like:
   ```markdown
   ![Tests](https://github.com/your-user/your-repo/actions/workflows/tests.yml/badge.svg)
   ```

5. **Add it to the top of your `README.md`**, commit, and push:
   ```bash
   git add README.md
   git commit -m "docs: add CI status badge"
   git push
   ```

6. **View your README on GitHub** — you should see a green "passing" badge (or red "failing" if tests are broken).

7. **Ask your AI agent:**

   > "What other badges are commonly used in Python project READMEs?"

   You'll likely learn about badges for: coverage, PyPI version, license, Python version support, and downloads.

**Expected outcome:** Your README displays a live build status badge, and you know about other common badges for Python projects.

## Publishing to PyPI

```yaml
name: Publish

on:
  release:
    types: [published]

jobs:
  publish:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: pip install build twine
      - run: python -m build
      - run: twine upload dist/*
        env:
          TWINE_USERNAME: __token__
          TWINE_PASSWORD: ${{ secrets.PYPI_TOKEN }}
```

## Exercise: Set Up CI

1. Ask AI to generate a workflow for your package
2. Add it to `.github/workflows/`
3. Push and watch it run
4. Fix any failures
5. Add status badge to README

## Summary

| Concept | Purpose |
|---------|--------|
| Workflow | YAML file defining automation |
| Job | Set of steps that run together |
| Matrix | Test multiple configurations |
| Secrets | Store sensitive data |

## Tips and Tricks

- **Prompt: "Write a GitHub Actions workflow that runs pytest on push"**: AI generates valid YAML for common CI patterns.
- **Start with the simplest workflow**: Test on one Python version, one OS — add matrix testing later when you need it.
- **Use `act` to test workflows locally**: `brew install act && act` runs your workflows on your machine — saves push-and-wait cycles.
- **Prompt: "Debug this GitHub Actions error: [paste log]"**: CI logs are verbose — AI is good at finding the relevant failure in the noise.
- **Cache dependencies**: Add `actions/cache` for pip — it can cut workflow time in half and costs nothing to set up.
- **Prompt: "Add a step to upload test coverage to [service]"**: AI knows the YAML for codecov, coveralls, etc.
- **Pin action versions**: Use `actions/checkout@v4`, not `actions/checkout@main` — pinned versions prevent surprise breakages.
- **Keep secrets in GitHub Settings**: Never hardcode tokens in workflow files; use `${{ secrets.MY_TOKEN }}` and add the secret in the repo settings.

## Foundational Concepts to Commit to Memory

- **CI (Continuous Integration)** = automatically running tests every time code is pushed, so broken code is caught immediately rather than discovered later
- **Workflow file** = a YAML file in `.github/workflows/` that defines what GitHub Actions should do and when
- **Trigger events** = `on: push`, `on: pull_request`, `on: release` — these control when your workflow runs
- **`actions/checkout@v4`** = the step that clones your repo into the runner; almost every workflow starts with this
- **`actions/setup-python@v5`** = sets up a specific Python version on the runner; always quote version numbers like `"3.10"` in YAML to avoid them being interpreted as floats
- **Matrix strategy** = run the same job across multiple configurations (Python versions, operating systems) in parallel; catches compatibility issues automatically
- **Secrets** = encrypted environment variables stored in your GitHub repo settings; use `${{ secrets.NAME }}` to reference them in workflows; never hardcode credentials
- **Status badges** = live indicators in your README showing whether CI is passing; they build trust and signal project health

## Knowing vs. Doing: Reflection

CI/CD is one of those topics where AI can write 90% of the YAML for you — and that is genuinely useful, because YAML workflow syntax is fiddly and hard to memorize. But here is the catch: if you do not understand what a matrix strategy does, or why you need `actions/checkout` before anything else, you will not know whether the AI-generated workflow is correct. You will push it, watch it fail, and have no idea why. The foundational knowledge is what lets you debug the output, not just generate it.

The practical reality is that most scientists and engineers do not need to become CI/CD experts. You need to know enough to set up a basic test workflow, read the logs when it fails, and add a matrix build when you need cross-version testing. That is a few hours of learning, not a few weeks. AI handles the boilerplate — the exact YAML syntax, the action version numbers, the secret configuration steps — while you handle the strategy: what should be tested, when, and on what platforms.

Think about the value you are creating. When you set up CI for your project, the value is not the YAML file itself — it is the confidence that your code works, visible to every collaborator via a green badge. That confidence compounds over time. Every future PR gets automatically tested. Every contributor knows immediately if they broke something. You invested a small amount of learning to create ongoing value for everyone who touches the project. That is the kind of leverage worth pursuing, and AI agents make the initial setup cost even lower.

## Additional Resources

- [GitHub Actions Documentation](https://docs.github.com/en/actions) — Official reference for workflows, syntax, and available actions
- [GitHub Actions Features](https://github.com/features/actions) — Overview of what GitHub Actions can do and how it fits into your development workflow

## Next Steps

In the next module, we'll learn AI-assisted documentation.